<a href="https://colab.research.google.com/github/Joh-Ishimwe/AmaliTech-DEG-Project-based-challenges/blob/main/AmaliTech_DEG_Veridi_Logistics_Josiane_Ishimwe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veridi Logistics — Delivery Performance Audit
**Olist Brazilian E-Commerce Dataset**  
Stories covered: 1 (Schema), 2 (Delay Calculator), 3 (Geographic Heatmap), 4 (Sentiment Correlation), Bonus (English Translation), Candidate's Choice

---

## 0 · Download Dataset
> Run this cell once to pull the Olist dataset from Kaggle.

In [20]:
# Using kagglehub to ensure reproducibility in a cloud environment.
# Subsequent runs use the cached download — no re-download needed.
import kagglehub

path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Dataset downloaded to:", path)


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset downloaded to: /kaggle/input/brazilian-ecommerce


## 1 · Install & Import Libraries

In [21]:
# plotly is not pre-installed in all Kaggle/Colab environments.
!pip install plotly --quiet

# ─ Standard imports ─
import pandas as pd
import numpy as np
import os
import json
import urllib.request
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')   # suppress non-critical pandas warnings

# Show all columns when displaying a dataframe (useful for wide tables)
pd.set_option('display.max_columns', None)

print("All libraries ready!")


All libraries ready!


## 2 · Explore Dataset Files

In [22]:
# List every file in the dataset folder with its size.
# This confirms the download completed and shows which CSVs are available.

dataset_path = "/kaggle/input/brazilian-ecommerce"

print("Files in dataset folder:")
for f in sorted(os.listdir(dataset_path)):
    size_mb = os.path.getsize(os.path.join(dataset_path, f)) / (1024 * 1024)
    print(f"  {f:55s} {size_mb:.2f} MB")


Files in dataset folder:
  olist_customers_dataset.csv                             8.62 MB
  olist_geolocation_dataset.csv                           58.44 MB
  olist_order_items_dataset.csv                           14.72 MB
  olist_order_payments_dataset.csv                        5.51 MB
  olist_order_reviews_dataset.csv                         13.78 MB
  olist_orders_dataset.csv                                16.84 MB
  olist_products_dataset.csv                              2.27 MB
  olist_sellers_dataset.csv                               0.17 MB
  product_category_name_translation.csv                   0.00 MB


## 3 · Story 1 — Schema Builder: Load & Inspect Raw Tables
Load the 6 required CSV files and verify row/column counts.

In [23]:
# Load only the 6 files required by the project brief.
# Note: full CSVs are loaded here; dtype optimisation can be added
# for very memory-constrained environments if needed.

dataset_path = "/kaggle/input/brazilian-ecommerce"

orders    = pd.read_csv(f"{dataset_path}/olist_orders_dataset.csv")
reviews   = pd.read_csv(f"{dataset_path}/olist_order_reviews_dataset.csv")
customers = pd.read_csv(f"{dataset_path}/olist_customers_dataset.csv")
products  = pd.read_csv(f"{dataset_path}/olist_products_dataset.csv")
items     = pd.read_csv(f"{dataset_path}/olist_order_items_dataset.csv")
translate = pd.read_csv(f"{dataset_path}/product_category_name_translation.csv")

# Sanity check: print shape of every table.
# If any row count is 0, the file path is wrong or the file is empty.
for name, df in [("orders", orders), ("reviews", reviews),
                 ("customers", customers), ("products", products),
                 ("items", items), ("translate", translate)]:
    print(f"{name:12s}  →  {df.shape[0]:>6,} rows  ×  {df.shape[1]:>2} cols")


orders        →  99,441 rows  ×   8 cols
reviews       →  99,224 rows  ×   7 cols
customers     →  99,441 rows  ×   5 cols
products      →  32,951 rows  ×   9 cols
items         →  112,650 rows  ×   7 cols
translate     →      71 rows  ×   2 cols


## 4 · Story 1 — Schema Builder: Prepare Tables for Joining
Before merging, we handle two sources of row duplication:
- **items**: one order can contain multiple items → keep only the first item per order.
- **reviews**: a few orders have duplicate review entries → keep only the most recent.

In [24]:
# ── SECTION 2A: De-duplicate the items table ─────────────────────────────────
# The items table has one row per product in each order.
# An order with 3 items has 3 rows. If we join this directly to orders,
# each order would be tripled — a classic fan-out bug.
# Fix: keep only ONE row per order (the first item listed).

items_first = items.drop_duplicates(subset="order_id", keep="first")

print(f"items (all rows):            {len(items):>7,}")
print(f"items_first (1 per order):   {len(items_first):>7,}")
print(f"orders:                      {len(orders):>7,}")
print("items_first ≈ orders count — row counts should be close ✓")

# ── SECTION 2B: De-duplicate the reviews table ────────────────────────────────
# A small number of orders received two review entries (e.g. the customer
# edited their review). We keep the most recent one by sorting descending
# on the answer timestamp before dropping duplicates.

reviews_clean = (
    reviews
    .sort_values("review_answer_timestamp", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
)

print(f"\nreviews (original):        {len(reviews):>7,}")
print(f"reviews (clean):           {len(reviews_clean):>7,}")
print(f"Duplicate reviews removed: {len(reviews) - len(reviews_clean):,}")


items (all rows):            112,650
items_first (1 per order):    98,666
orders:                       99,441
items_first ≈ orders count — row counts should be close ✓

reviews (original):         99,224
reviews (clean):            98,673
Duplicate reviews removed: 551


## 5 · Story 1 — Schema Builder: Build Master DataFrame
Chain five left-joins to combine all tables into a single analysis-ready dataset.

In [25]:
# ── SECTION 2C: Join all tables → master dataframe ───────────────────────────
#
# Join map (each arrow = one merge on the stated key):
#
#   orders ──(order_id)──────► reviews_clean   adds: review_score
#          ──(customer_id)───► customers        adds: customer_state, customer_city
#          ──(order_id)──────► items_first      adds: product_id
#          ──(product_id)────► products         adds: product_category_name (PT)
#          ──(category name)► translate         adds: product_category_name_english
#
# how="left" keeps ALL orders even when a matching row is absent in the right
# table (e.g. an order with no review gets NaN for review_score, not dropped).

master = (
    orders
    .merge(reviews_clean[["order_id", "review_score"]],
           on="order_id", how="left")
    .merge(customers[["customer_id", "customer_state", "customer_city"]],
           on="customer_id", how="left")
    .merge(items_first[["order_id", "product_id"]],
           on="order_id", how="left")
    .merge(products[["product_id", "product_category_name"]],
           on="product_id", how="left")
    .merge(translate, on="product_category_name", how="left")
)

# ── Join integrity check ──────────────────────────────────────────────────────
# After a left join the row count must equal the left table's row count.
# Any increase means a 1-to-many join snuck through and duplicated rows.
assert len(master) == len(orders), (
    f"Row count mismatch! orders={len(orders):,}, master={len(master):,}. "
    "A join produced duplicates — investigate."
)

print(f"orders:  {len(orders):>7,} rows  ← source")
print(f"master:  {len(master):>7,} rows  ← must match ✓")
print(f"master columns ({master.shape[1]} total): {list(master.columns)}")


orders:   99,441 rows  ← source
master:   99,441 rows  ← must match ✓
master columns (14 total): ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'review_score', 'customer_state', 'customer_city', 'product_id', 'product_category_name', 'product_category_name_english']


## 6 · Exploratory Data Analysis (EDA)
Before cleaning, we audit the data: nulls, duplicates, data types, and distribution of key columns.

> **Note on data types:** Date columns appear as `object` (text) in section 4 and 5 below — this is expected. They are converted to proper `datetime64` types in Section 7. As a result, `describe()` only summarises `review_score` at this stage.

In [26]:
# ── SECTION 2.5: EDA — 'interview' the master dataframe ─────────────────────

print("=" * 60)
print("1. SAMPLE ROWS")
print("=" * 60)
display(master.head(3))

# ── Missing values ────────────────────────────────────────────────────────────
# High null % signals a column needs special handling: impute, flag, or drop.
print("\n" + "=" * 60)
print("2. MISSING VALUES (columns with nulls only)")
print("=" * 60)
null_counts = master.isnull().sum()
null_pct    = (null_counts / len(master) * 100).round(2)
null_df     = pd.DataFrame({"missing_count": null_counts, "missing_%": null_pct})
print(null_df[null_df["missing_count"] > 0].to_string())

# ── Duplicate rows ────────────────────────────────────────────────────────────
# Confirms the de-duplication steps in Section 4 worked correctly.
print("\n" + "=" * 60)
print("3. DUPLICATE ROWS (based on order_id)")
print("=" * 60)
duplicate_rows = master[master.duplicated(subset='order_id', keep=False)]
if not duplicate_rows.empty:
    print(f"Found {len(duplicate_rows):,} duplicate rows based on order_id. Displaying first 5:\n")
    display(duplicate_rows.head())
else:
    print("No duplicate rows found based on order_id. ✓")

# ── Data types ────────────────────────────────────────────────────────────────
# Date columns show as 'object' (text) here — fixed in Section 7.
print("\n" + "=" * 60)
print("4. DATA TYPES")
print("=" * 60)
print(master.dtypes.to_string())

# ── Descriptive statistics ────────────────────────────────────────────────────
# Only review_score is numeric at this stage; date columns are still text.
print("\n" + "=" * 60)
print("5. DESCRIPTIVE STATISTICS (Numerical Columns)")
print("=" * 60)
display(master.describe())

# ── Order status breakdown ────────────────────────────────────────────────────
# Only 'delivered' orders have an actual delivery date; others are excluded
# from delay calculations in Section 7.
print("\n" + "=" * 60)
print("6. ORDER STATUS BREAKDOWN")
print("=" * 60)
status_counts = master["order_status"].value_counts()
status_pct    = (status_counts / len(master) * 100).round(2)
print(pd.DataFrame({"count": status_counts, "%": status_pct}).to_string())

# ── Review score distribution ─────────────────────────────────────────────────
# Expected range: 1–5 integers. Any value outside that range is a data error.
print("\n" + "=" * 60)
print("7. REVIEW SCORE DISTRIBUTION")
print("=" * 60)
print(master["review_score"].value_counts().sort_index().to_string())


1. SAMPLE ROWS


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,customer_state,customer_city,product_id,product_category_name,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,4.0,SP,sao paulo,87285b34884572647811a353c7ac498a,utilidades_domesticas,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,4.0,BA,barreiras,595fac2a385ac33a80bd5114aec74eb8,perfumaria,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,5.0,GO,vianopolis,aa4383b373c6aca5d8797843e5594415,automotivo,auto



2. MISSING VALUES (columns with nulls only)
                               missing_count  missing_%
order_approved_at                        160       0.16
order_delivered_carrier_date            1783       1.79
order_delivered_customer_date           2965       2.98
review_score                             768       0.77
product_id                               775       0.78
product_category_name                   2191       2.20
product_category_name_english           2212       2.22

3. DUPLICATE ROWS (based on order_id)
No duplicate rows found based on order_id. ✓

4. DATA TYPES
order_id                          object
customer_id                       object
order_status                      object
order_purchase_timestamp          object
order_approved_at                 object
order_delivered_carrier_date      object
order_delivered_customer_date     object
order_estimated_delivery_date     object
review_score                     float64
customer_state                    objec

,review_score
count,98673.000000
mean,4.086386
std,1.347618
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000



6. ORDER STATUS BREAKDOWN
              count      %
order_status              
delivered     96478  97.02
shipped        1107   1.11
canceled        625   0.63
unavailable     609   0.61
invoiced        314   0.32
processing      301   0.30
created           5   0.01
approved          2   0.00

7. REVIEW SCORE DISTRIBUTION
review_score
1.0    11363
2.0     3131
3.0     8133
4.0    19038
5.0    57008


### 6a · Review Score Distribution Chart
A visual breakdown of how customers rated their orders overall.

In [37]:
review_score_distribution = master['review_score'].value_counts().sort_index()

fig_review_dist = px.bar(
    x=review_score_distribution.index,
    y=review_score_distribution.values,
    labels={'x': 'Review Score', 'y': 'Number of Orders'},
    title='Distribution of Review Scores (1–5)',
    color=review_score_distribution.index,
    color_continuous_scale=px.colors.sequential.Viridis,
    text=review_score_distribution.values # Add text labels
)
fig_review_dist.update_traces(texttemplate='%{text}', textposition='outside') # Position text outside bars
fig_review_dist.update_layout(
    xaxis_title="Review Score (1–5)",
    yaxis_title="Number of Reviews",
    xaxis_type='category',
    coloraxis_showscale=False,
)
fig_review_dist.show()

print("Review score distribution chart generated successfully.")

Review score distribution chart generated successfully.


> **Insight:** Score 5 dominates with 57,008 reviews, over half of all ratings.
> Score 1 is a distant but notable third (11,363), showing that when the experience
> goes wrong, customers react strongly. Scores 2 and 3 are the least common,
> meaning customers rarely feel "neutral", delivery quality pushes them to either
> love or hate the experience.

## 7 · Story 2 — Data Cleaning & Delay Calculation
Convert date columns, filter to delivered orders, calculate `days_difference`, and classify delay severity.

In [28]:
# ── SECTION 3A: Convert date columns from text → datetime ────────────────────
# pandas reads all CSV columns as text by default.
# We must convert date columns so Python can do date arithmetic.
# errors='coerce' turns unparseable values into NaT instead of crashing.

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    master[col] = pd.to_datetime(master[col], errors="coerce")

print("Date column dtypes after conversion:")
print(master[date_columns].dtypes)

# ── SECTION 3B: Filter to delivered orders only ───────────────────────────────
# Only orders with status='delivered' have an actual delivery date.
# Canceled/unavailable orders have NaT in order_delivered_customer_date
# and would produce NaN delays — we exclude them entirely.
# .copy() creates an independent DataFrame (avoids SettingWithCopyWarning).

delivered = master[master["order_status"] == "delivered"].copy()

print(f"\nTotal orders:     {len(master):>7,}")
print(f"Delivered orders: {len(delivered):>7,}  ({len(delivered)/len(master)*100:.1f}% of total)")

# ── SECTION 3C: Calculate days_difference ─────────────────────────────────────
# days_difference = estimated_delivery − actual_delivery
#   Positive → arrived EARLY  (good: we over-delivered on our promise)
#   Zero     → arrived EXACTLY on the promised date
#   Negative → arrived LATE   (bad: we broke our promise to the customer)
# .dt.days converts the Timedelta result to a plain integer.

delivered["days_difference"] = (
    delivered["order_estimated_delivery_date"] -
    delivered["order_delivered_customer_date"]
).dt.days

# ── SECTION 3D: Classify delivery performance ─────────────────────────────────
# On Time   : days_difference >= 0    (on time or early)
# Late      : -5 <= days_diff < 0     (1–5 days past the promised date)
# Super Late: days_difference < -5    (more than 5 days past promised date)
# Unknown   : NaN days_difference     (missing date — edge case)

def classify_delay(days):
    """Return a delivery status label based on days_difference."""
    if pd.isna(days):
        return "Unknown"
    elif days >= 0:
        return "On Time"
    elif days >= -5:
        return "Late"
    else:
        return "Super Late"

delivered["delivery_status"] = delivered["days_difference"].apply(classify_delay)

# ── SECTION 3E: Fill missing English category names ───────────────────────────
# A small number of orders have no product category in the source data.
# We label them 'Unknown' so they remain in geographic analysis.

delivered["product_category_name_english"] = (
    delivered["product_category_name_english"].fillna("Unknown")
)

# ── FIX: use .copy() to prevent SettingWithCopyWarning downstream ─────────────
# reviewed is used in Section 9 to add a 'delay_bucket' column.
# Without .copy(), pandas may warn about modifying a slice.
reviewed = delivered.dropna(subset=["review_score"]).copy()

# ── Summary ───────────────────────────────────────────────────────────────────
print("\nDelivery status breakdown:")
status_summary = delivered["delivery_status"].value_counts()
status_pct     = (status_summary / len(delivered) * 100).round(1)
print(pd.DataFrame({"count": status_summary, "%": status_pct}))

print(f"\ndays_difference range: "
      f"{delivered['days_difference'].min():.0f} to "
      f"{delivered['days_difference'].max():.0f} days")

print("\nSample of key engineered columns:")
display(delivered[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "days_difference",
    "delivery_status",
]].head(5))


Date column dtypes after conversion:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Total orders:      99,441
Delivered orders:  96,478  (97.0% of total)

Delivery status breakdown:
                 count     %
delivery_status             
On Time          88644  91.9
Super Late        4211   4.4
Late              3615   3.7
Unknown              8   0.0

days_difference range: -189 to 146 days

Sample of key engineered columns:


,order_id,order_delivered_customer_date,order_estimated_delivery_date,days_difference,delivery_status
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,7.0,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,5.0,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,17.0,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,12.0,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,9.0,On Time


## 8 · Story 3 — Geographic Heatmap: Late Deliveries by State
Calculate the percentage of late + super-late orders per Brazilian state and visualise.

In [29]:
# ── SECTION 4: Late delivery rate by state ───────────────────────────────────
#
# Step 1: Tag each order as late (Late or Super Late) vs on time.
# Step 2: Group by state → compute total orders and late order count.
# Step 3: Calculate late_pct = late_orders / total_orders * 100.
# Step 4: Plot a sorted bar chart.

# Binary flag: 1 = late or super late, 0 = on time
delivered["is_late"] = delivered["delivery_status"].isin(
    ["Late", "Super Late"]
).astype(int)

# Aggregate by state
state_stats = (
    delivered
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        late_orders=("is_late", "sum"),
    )
    .reset_index()
)

# Calculate late % and sort worst-to-best
state_stats["late_pct"] = (
    state_stats["late_orders"] / state_stats["total_orders"] * 100
).round(1)
state_stats = state_stats.sort_values("late_pct", ascending=False)

# ── Bar chart ────────────────────────────────────────────────────────────────
fig_state = px.bar(
    state_stats,
    x="customer_state",
    y="late_pct",
    color="late_pct",
    color_continuous_scale="Reds",
    title="Late Delivery Rate (%) by Brazilian State",
    labels={"customer_state": "State", "late_pct": "Late Delivery Rate (%)"},
    text="late_pct",
)
fig_state.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_state.update_layout(
    xaxis_title="State (sorted worst → best)",
    yaxis_title="% Late Orders",
    coloraxis_showscale=False,
    height=500,
)
fig_state.show()

# ── Print national average and top 3 worst states ─────────────────────────────
national_avg = delivered["is_late"].sum() / len(delivered) * 100
print(f"National late delivery rate: {national_avg:.1f}%")
print("\nTop 3 worst-performing states:")
print(state_stats.head(3)[["customer_state", "late_pct", "total_orders"]].to_string(index=False))


National late delivery rate: 8.1%

Top 3 worst-performing states:
customer_state  late_pct  total_orders
            AL      23.9           397
            MA      19.7           717
            PI      16.0           476


> **Insight:** AL leads at 23.9% late rate, more than 8× worse than the
> best-performing state RO (2.9%). The top 5 worst states (AL, MA, PI, CE, SE)
> are all in the North or Northeast, while SP (5.6%), the home of Olist's main
> distribution hub, sits near the bottom. Distance from the hub is the clearest
> predictor of late delivery rate.

### 8a · Choropleth Map: Late Delivery Rate by State
A map view reinforces the geographic concentration of late deliveries.

In [30]:
# ── Choropleth map of Brazil coloured by late delivery rate ──────────────────
#
# We fetch the GeoJSON for Brazilian state boundaries from a public source.
# featureidkey must match the property name used for state abbreviations
# in that GeoJSON file. After inspection, the correct key is 'sigla' — NOT
# 'UF_05' (which does not exist in this dataset and would render a blank map).

geojson_url = (
    "https://raw.githubusercontent.com/codeforamerica/click_that_hood"
    "/master/public/data/brazil-states.geojson"
)
with urllib.request.urlopen(geojson_url) as url:
    brazil_states_geojson = json.loads(url.read().decode())

# Verify the correct property key by inspecting the first feature
first_props = brazil_states_geojson["features"][0]["properties"]
print("GeoJSON feature properties:", list(first_props.keys()))
print("Sample sigla value:", first_props.get("sigla", "KEY NOT FOUND"))

fig_choropleth = px.choropleth_mapbox(
    state_stats,
    geojson=brazil_states_geojson,
    locations="customer_state",
    featureidkey="properties.sigla",   # ← correct key in this GeoJSON
    color="late_pct",
    color_continuous_scale="Reds",
    range_color=(state_stats["late_pct"].min(), state_stats["late_pct"].max()),
    mapbox_style="carto-positron",
    zoom=3.5,
    center={"lat": -14.235, "lon": -53.18},
    opacity=0.7,
    labels={"customer_state": "State", "late_pct": "Late Delivery Rate (%)"},
    title="Late Delivery Rate (%) by Brazilian State",
)
fig_choropleth.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0})
fig_choropleth.show()

print("Choropleth map generated successfully.")


GeoJSON feature properties: ['id', 'name', 'sigla', 'regiao_id', 'codigo_ibg', 'cartodb_id', 'created_at', 'updated_at']
Sample sigla value: AC


Choropleth map generated successfully.


> **Insight:** The map confirms what the bar chart shows numerically the
> dark-red cluster sits entirely in the Northeast coastline. The further a state
> is from São Paulo, the deeper its shade. A Regional Director can point to this
> map to justify targeted investment in northern carrier contracts or
> micro-warehousing rather than a nationwide programme.

## 9 · Story 4 — Sentiment Correlation: Does Lateness Cause Bad Reviews?
Compare average review scores across delivery status categories and across the delay spectrum.

In [38]:
# ── SECTION 5A: Mean review score per delivery status ────────────────────────
# 'reviewed' was defined in Section 7 as delivered orders that have a review score.
# .copy() was applied there, so assigning new columns here is safe.

sentiment_by_status = (
    reviewed
    .groupby("delivery_status")
    .agg(
        avg_review=("review_score", "mean"),
        order_count=("order_id", "count"),
    )
    .reset_index()
    .sort_values("avg_review", ascending=False)
)
sentiment_by_status["avg_review"] = sentiment_by_status["avg_review"].round(2)

print("Average review score by delivery status:")
print(sentiment_by_status.to_string(index=False))

# ── Bar chart: average review per delivery status ─────────────────────────────
fig_sentiment = px.bar(
    sentiment_by_status,
    x="delivery_status",
    y="avg_review",
    color="delivery_status",
    color_discrete_map={
        "On Time":    "#2ecc71",
        "Late":       "#e67e22",
        "Super Late": "#e74c3c",
        "Unknown":    "#95a5a6",
    },
    title="Average Review Score by Delivery Status",
    labels={"delivery_status": "Delivery Status", "avg_review": "Avg Review Score (1–5)"},
    text="avg_review",
)
fig_sentiment.update_traces(texttemplate="%{text}", textposition="outside")
fig_sentiment.update_layout(yaxis_range=[0, 5.5], showlegend=False, height=450)

fig_sentiment = px.bar(
    sentiment_by_status,
    x="delivery_status",
    y="avg_review",
    color="delivery_status",
    color_discrete_map={
        "On Time":    "#2ecc71",
        "Late":       "#e67e22",
        "Super Late": "#e74c3c",
        "Unknown":    "#95a5a6",
    },
    title="Average Review Score by Delivery Status",
    labels={"delivery_status": "Delivery Status", "avg_review": "Avg Review Score (1–5)"},
    text="avg_review",
    hover_data={"order_count": True},   # ← shows count on hover
)
fig_sentiment.update_traces(texttemplate="%{text}", textposition="outside")
fig_sentiment.update_layout(yaxis_range=[0, 5.5], showlegend=False, height=450)

# Add order count as a subtitle annotation under each bar
for _, row in sentiment_by_status.iterrows():
    fig_sentiment.add_annotation(
        x=row["delivery_status"],
        y=0.15,
        text=f"n = {row['order_count']:,}",
        showarrow=False,
        font=dict(size=11, color="#555"),
        yref="y",
    )

fig_sentiment.show()


Average review score by delivery status:
delivery_status  avg_review  order_count     display_text
        Unknown        4.50            8       4.5<br>n=8
        On Time        4.29        88163 4.29<br>n=88,163
           Late        3.46         3568  3.46<br>n=3,568
     Super Late        1.78         4093  1.78<br>n=4,093


> **Insight:** On-time deliveries score 4.29/5, while Super Late
> orders collapse to 1.78/5, a 58% drop in satisfaction. Even
> moderately late orders (Late, 3.46/5) lose nearly a full point vs on-time.
> The Unknown bar (n=8) is too small to be meaningful and can be ignored.
> This directly answers the CEO's question: logistics failure is measurably
> destroying customer sentiment.

In [32]:
# ── SECTION 5B: Average review score per delay-day bucket ────────────────────
# Bin days_difference into labelled brackets to show the continuous trend.
# pd.cut() divides a numeric column into discrete intervals.

bins   = [-999, -30, -14, -7, -5, -1, 0, 7, 14, 999]
labels = [">30d Late", "15–30d Late", "8–14d Late",
          "6–7d Late",  "3–5d Late",  "1–2d Late",
          "On Time",    "1–7d Early", ">7d Early"]

reviewed["delay_bucket"] = pd.cut(
    reviewed["days_difference"],
    bins=bins,
    labels=labels,
)

bucket_avg = (
    reviewed
    .groupby("delay_bucket", observed=True)["review_score"]
    .mean()
    .reset_index()
    .rename(columns={"review_score": "avg_review"})
)
bucket_avg["avg_review"] = bucket_avg["avg_review"].round(2)

fig_trend = px.line(
    bucket_avg,
    x="delay_bucket",
    y="avg_review",
    markers=True,
    title="Average Review Score vs Delivery Delay Bucket",
    labels={"delay_bucket": "Delivery Delay Bracket", "avg_review": "Avg Review Score"},
)
fig_trend.update_layout(yaxis_range=[0, 5.5], height=450)
fig_trend.show()


> **Insight:** The sharpest drop in satisfaction occurs between **3–5 days late** around(3.65) and **6–7 days late** (~2.3) — that is the critical threshold where customer
> tolerance breaks down. Beyond 8 days late, scores floor around 1.75–2.0 regardless
> of how much later the package arrives. Interestingly, 1–2 days late still scores
> ~4.2 — almost as high as on time — meaning customers are forgiving of very minor
> delays.

### 9a · Scatter Plot: Review Score vs. Days Difference

In [33]:
# Create a scatter plot of review_score vs. days_difference
# To make the plot readable, we'll sample a subset of the data if it's too large.
# Also, adding jitter to `review_score` can help visualize density where points overlap.

# Consider sampling if 'reviewed' is very large for performance
plot_data = reviewed.sample(n=min(10000, len(reviewed)), random_state=42) if len(reviewed) > 10000 else reviewed

fig_scatter_sentiment = px.scatter(
    plot_data,
    x="days_difference",
    y="review_score",
    title="Review Score vs. Days Difference (Sampled Data)",
    labels={
        "days_difference": "Days Difference (Estimated - Actual)",
        "review_score": "Review Score"
    },
    hover_data=['order_id', 'delivery_status'],
    color="review_score",
    color_continuous_scale="RdYlGn" # Red for low scores, Green for high scores
)

fig_scatter_sentiment.update_layout(yaxis_range=[0.5, 5.5]) # Ensure review scores 1-5 are clearly visible
fig_scatter_sentiment.show()

print("Scatter plot of review score vs. days difference generated successfully.")


Scatter plot of review score vs. days difference generated successfully.


## 10 · Bonus + Candidate's Choice — Category Performance Analysis
**Bonus Story**: English category names are already applied via `product_category_name_translation.csv`.

**Candidate's Choice**: Identify which product categories have the highest late delivery rates and the lowest average review scores — giving the logistics team a prioritised fix list.

> **Business justification**: Not all lateness is equal. If furniture is 3× harder to ship on time than electronics, the operations team should negotiate different SLAs or carrier contracts for heavy/bulky categories before rolling out a blanket improvement programme.

In [34]:
# ── SECTION 6: Category-level delivery performance ───────────────────────────
# Group by English category name, compute late % and avg review.
# Filter out 'Unknown' to keep the chart clean.
# Minimum 100 orders per category to ensure statistical reliability.

cat_stats = (
    delivered
    .dropna(subset=["review_score"])
    .query("product_category_name_english != 'Unknown'")
    .groupby("product_category_name_english")
    .agg(
        total_orders=("order_id", "count"),
        late_orders=("is_late", "sum"),
        avg_review=("review_score", "mean"),
    )
    .reset_index()
)

cat_stats = cat_stats[cat_stats["total_orders"] >= 100].copy()
cat_stats["late_pct"]   = (cat_stats["late_orders"] / cat_stats["total_orders"] * 100).round(1)
cat_stats["avg_review"] = cat_stats["avg_review"].round(2)
cat_stats = cat_stats.sort_values("late_pct", ascending=False)

# ── Chart: Top 15 categories by late delivery rate ────────────────────────────
# Bar height = late %; bar colour = avg review (red = low, green = high).
# A tall red bar = high lateness AND low satisfaction — the worst offenders.

fig_cat = px.bar(
    cat_stats.head(15),
    x="product_category_name_english",
    y="late_pct",
    color="avg_review",
    color_continuous_scale="RdYlGn",
    range_color=[1, 5],
    title="Top 15 Product Categories: Late Delivery Rate vs Avg Review Score",
    labels={
        "product_category_name_english": "Category (English)",
        "late_pct": "Late Delivery Rate (%)",
        "avg_review": "Avg Review Score",
    },
    text="late_pct",
)
fig_cat.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_cat.update_layout(
    xaxis_tickangle=-45,
    height=550,
    coloraxis_colorbar_title="Avg Review",
)
fig_cat.show()

print("Top 10 worst categories (late delivery + review score):")
print(
    cat_stats.head(10)
    [["product_category_name_english", "total_orders", "late_pct", "avg_review"]]
    .to_string(index=False)
)


Top 10 worst categories (late delivery + review score):
product_category_name_english  total_orders  late_pct  avg_review
                        audio           341      12.9        3.85
      fashion_underwear_beach           116      12.9        4.01
              books_technical           254      11.0        4.43
                 home_confort           368      10.3        3.91
                         food           431      10.0        4.32
                  electronics          2488       9.8        4.13
                         baby          2741       9.2        4.12
    construction_tools_lights           228       9.2        4.21
             office_furniture          1236       9.1        3.65
          musical_instruments           601       9.0        4.25


> **Insight:** Category-level analysis reveals that lateness is not evenly distributed across product types. Categories like **audio**, **fashion_underwear_beach**, and **home_comfort** combine high late delivery rates with the lowest average review scores (dark red bars) — making them the highest-priority targets for operational review. Interestingly, some high-lateness categories still maintain moderate review scores, suggesting customers may be more forgiving for certain product types (e.g. non-urgent items). This nuance means a one-size-fits-all SLA policy would miss the real problem: the operations team should focus first on the **red + tall** bars.

## 11 · Export Clean Dataset
Save the final `delivered` DataFrame as a CSV for use in your dashboard tool.

In [35]:
# ── Export the cleaned, enriched dataset ─────────────────────────────────────
# Select only the columns needed for dashboarding.
# Relative path ensures reproducibility (no local machine paths).

export_cols = [
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "days_difference",
    "delivery_status",
    "customer_state",
    "customer_city",
    "review_score",
    "product_category_name_english",
]

output_path = "veridi_delivery_audit_clean.csv"
delivered[export_cols].to_csv(output_path, index=False)

print(f"Exported {len(delivered):,} rows × {len(export_cols)} cols → {output_path}")
print("\nColumn overview:")
print(delivered[export_cols].dtypes)


Exported 96,478 rows × 10 cols → veridi_delivery_audit_clean.csv

Column overview:
order_id                                 object
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
days_difference                         float64
delivery_status                          object
customer_state                           object
customer_city                            object
review_score                            float64
product_category_name_english            object
dtype: object


## 12 · Conclusion

This delivery performance audit of the Olist Brazilian E-Commerce dataset has provided several key insights:

1. **Late Deliveries are a Significant Issue:** While the majority of orders are delivered on time, a non-trivial percentage experience delays. This is particularly evident in certain states, with **Alagoas (AL)**, **Maranhão (MA)**, and **Piauí (PI)** showing significantly higher late delivery rates compared to the national average — consistent with their distance from São Paulo's distribution hubs.

2. **Delivery Delay Directly Impacts Customer Satisfaction:** There is a clear and strong negative correlation between delivery lateness and customer review scores. Orders classified as 'Super Late' received drastically lower average review scores (~1.8) compared to 'On Time' deliveries (~4.3). Even 'Late' deliveries saw a notable drop in satisfaction. Exceeding the promised date — arriving early — yields the *highest* scores, suggesting conservative ETA-setting as a low-cost improvement lever.

3. **Product Categories Have Varying Delivery Performance:** Categories like `audio`, `fashion_underwear_beach`, and `home_comfort` combine high late delivery rates with the lowest average review scores, suggesting category-specific logistics bottlenecks that warrant targeted carrier negotiations or routing changes.

4. **Data Quality and Engineering:** The process involved de-duplication of both the items and reviews tables, date-type conversion, and multi-table joining — all critical for accurate delay calculation. The review score distribution shows a bimodal pattern (most common scores: 5 and 1), confirming that the delivery experience is the dominant driver of extreme customer sentiment.

**Recommendations:**

- **Targeted State-Level Interventions:** Prioritise AL, MA, and PI for dedicated carrier contracts or regional micro-warehousing.
- **Conservative ETA Setting:** Where possible, under-promise on estimated delivery dates to convert 'on time' deliveries into 'early' — the highest-scoring category.
- **Category-Specific Logistics Review:** Investigate the supply chain for audio, home comfort, and fashion categories to identify specific bottlenecks before applying blanket policy changes.

These findings provide actionable intelligence for Veridi Logistics to improve delivery performance and, consequently, customer satisfaction.